In [1]:
import os, sys, torch

# --- ensure working directory is project root ---
PROJECT_ROOT = '/scratch/jq2uw/edit-skingpt4'
os.chdir(PROJECT_ROOT)
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
print(f'Working directory: {os.getcwd()}')

os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'max_split_size_mb:128'
torch.cuda.empty_cache()

from model_skingpt4 import *

# --- load raw pretrained SkinGPT-4 ---
print('Loading model...')
cfg = init_cfg(gpu_id=0)  # defaults to llama2_13bchat
model, vis_processor, chat = init_chat(cfg)
print(f'Trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}')
print(f'Total params:     {sum(p.numel() for p in model.parameters()):,}')

Working directory: /sfs/weka/scratch/jq2uw/edit-skingpt4


/home/jq2uw/miniconda3/envs/skingpt4/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading model...
Initializing Configs
Initializing Chat
Loading VIT


/home/jq2uw/miniconda3/envs/skingpt4/lib/python3.9/site-packages/huggingface_hub/file_download.py:945: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Loading VIT Done
Loading Q-Former
Loading Q-Former Done
Loading LLM tokenizer
Loading LLM model


Loading checkpoint shards: 100%|█████████████████████████████████████| 3/3 [00:21<00:00,  7.27s/it]


Loading LLM Done
Load 2 training prompts
Prompt Example 
###Human: <Img><ImageHere></Img> Could you describe the skin disease in this image for me? ###Assistant: 
Load BLIP2-LLM Checkpoint: ./model_skingpt4/weights/skingpt4_llama2_13bchat_base_pretrain_stage2.pth
Initialization Finished
Trainable params: 3,937,280
Total params:     14,110,861,184


In [2]:
import pandas as pd
from PIL import Image
from pathlib import Path

DATA_DIR = Path('./data')

# --- load shared dataset ---
df = pd.read_parquet('./data_share/midas_share.parquet')
print(f'Loaded {len(df)} rows, columns: {list(df.columns)}')
print(f'uid sample: {df["uid"].iloc[:3].tolist()}')
print(f'image_path raw sample: {df["image_path"].iloc[:3].tolist()}')
print(f'y3 distribution:\n{df["y3"].value_counts()}')

# --- resolve image paths: if not absolute / not found, look under ./data/ ---
def resolve_img_path(p):
    p = str(p)
    if os.path.isfile(p):
        return p
    candidate = DATA_DIR / Path(p).name  # try just the filename under ./data/
    if candidate.is_file():
        return str(candidate)
    return p  # return as-is, will be caught by try/except later

df['image_path_resolved'] = df['image_path'].apply(resolve_img_path)
n_found = df['image_path_resolved'].apply(os.path.isfile).sum()
print(f'\nResolved images: {n_found}/{len(df)} found under {DATA_DIR}')
print(f'Resolved sample: {df["image_path_resolved"].iloc[:3].tolist()}')

Loaded 3357 rows, columns: ['uid', 'patient_id', 'image_path', 'question', 'answer', 'rationale', 'choices', 'idx_choices', 'id_patient', 'y16_description', 'y16', 'y3', 'demo_gender', 'demo_age', 'demo_fitzpatrick_skintype', 'demo_melanoma_history', 'demo_ethnicity', 'demo_race', 'lesion_distance', 'lesion_location', 'lesion_length_mm', 'lesion_width_mm', 'notes_clinical_impression_1', 'notes_clinical_impression_2', 'notes_clinical_impression_3', 'notes_pathreport', 'x_skintype', 'x_skincolor', 'x_skintone', 'x_location', 'text_y3', 'text_y16', 'text_x_skintype', 'text_x_skincolor', 'text_x_skintone', 'text_x_location', 'text_demo', 'text_lesion']
uid sample: [1, 2, 3]
image_path raw sample: ['data/images/midas/s-prd-398966407.jpg', 'data/images/midas/s-prd-398966642.jpg', 'data/images/midas/s-prd-398966845.jpg']
y3 distribution:
y3
malignant    1391
benign       1322
other         644
Name: count, dtype: int64

Resolved images: 3357/3357 found under data
Resolved sample: ['data/s-prd

In [3]:
from tqdm import tqdm

question = 'Is the lesion malignant or benign, or other?'
results = []

print(f'Running predictions on {len(df)} images...')
print(f'Question: "{question}"')
print('-' * 60)

for i, row in tqdm(df.iterrows(), total=len(df)):
    uid = row['uid']
    img_path = row['image_path_resolved']

    try:
        image = Image.open(img_path).convert('RGB')
    except Exception as e:
        print(f'[SKIP] uid={uid}, cannot open {img_path}: {e}')
        continue

    response = chat_with_image(chat, image, question, temperature=0.0, remove_system=True)

    results.append({
        'uid': uid,
        'response': response,
    })

    if len(results) <= 3:  # print first few for sanity check
        print(f'[{len(results)}] uid={uid}  gt={row["y3"]}')
        print(f'    {response[:200]}')
        print()

print(f'Done. Collected {len(results)} predictions.')

Running predictions on 3357 images...
Question: "Is the lesion malignant or benign, or other?"
------------------------------------------------------------


  0%|                                                           | 1/3357 [00:05<4:59:24,  5.35s/it]

[1] uid=1  gt=malignant
    This is an image of a person's arm, showing redness and swelling due to allergy or insect bites. The skin appears thin and fragile, and there are small pimples and bumps on the surface. In addition, t



  0%|                                                           | 2/3357 [00:08<3:58:26,  4.26s/it]

[2] uid=2  gt=malignant
    The image shows a large, red and swollen mole on the arm of a person. The mole is covered in small, red and purple veins and appears to be inflamed. There are also several small, red and purple spots 



  0%|                                                           | 3/3357 [00:11<3:29:14,  3.74s/it]

[3] uid=3  gt=malignant
    This image appears to be a close up view of a woman's face with redness and inflammation on her skin, as well as small blood vessels visible under the skin. Her eyes are also visible, with a small amo



  3%|█▊                                                       | 105/3357 [05:50<3:00:44,  3.33s/it]


╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:20                                                                                   │
│                                                                                                  │
│   17 │   │   print(f'[SKIP] uid={uid}, cannot open {img_path}: {e}')                             │
│   18 │   │   continue                                                                            │
│   19 │                                                                                           │
│ ❱ 20 │   response = chat_with_image(chat, image, question, temperature=0.0, remove_system=Tru    │
│   21 │                                                                                           │
│   22 │   results.append({                                                                        │
│   23 │   │   'uid': uid,                                                                         │
│                                                                                                  │
│ /scratch/jq2uw/edit-skingpt4/model_skingpt4/__init__.py:86 in chat_with_image                    │
│                                                                                                  │
│   83 │   img_list = []                                                                           │
│   84 │   _ = chat.upload_img(image, chat_state, img_list)                                        │
│   85 │   chat.ask(question, chat_state)                                                          │
│ ❱ 86 │   response_dict = chat.answer(                                                            │
│   87 │   │   conv=chat_state,                                                                    │
│   88 │   │   img_list=img_list,                                                                  │
│   89 │   │   num_beams=num_beams,                                                                │
│                                                                                                  │
│ /home/jq2uw/miniconda3/envs/skingpt4/lib/python3.9/site-packages/torch/utils/_contextlib.py:115  │
│ in decorate_context                                                                              │
│                                                                                                  │
│   112 │   @functools.wraps(func)                                                                 │
│   113 │   def decorate_context(*args, **kwargs):                                                 │
│   114 │   │   with ctx_factory():                                                                │
│ ❱ 115 │   │   │   return func(*args, **kwargs)                                                   │
│   116 │                                                                                          │
│   117 │   return decorate_context                                                                │
│   118                                                                                            │
│                                                                                                  │
│ /scratch/jq2uw/edit-skingpt4/model_skingpt4/skingpt4/conversation/conversation.py:157 in answer  │
│                                                                                                  │
│   154 │   │   embs = embs[:, begin_idx:]                                                         │
│   155 │   │   do_sample = False if temperature < 1e-4 else True                                  │
│   156 │   │   temperature = 0.0 if temperature < 1e-4 else temperature                           │
│ ❱ 157 │   │   outputs = self.model.llm_model.generate(                                           │
│   158 │   │   │   inputs_embeds=embs,                                                            │
│   159 │   │   │   max_new_tokens=max_new_tokens,           

In [4]:
results

[{'uid': 1,
  'response': 'This is an image of a person\'s arm, showing redness and swelling due to allergy or insect bites. The skin appears thin and fragile, and there are small pimples and bumps on the surface. In addition, there are small scabs and scratch marks on the skin, indicating irritation or infection. Overall, it appears to be a mild case of eczema or dermatitis.\n###NLL:{"benign":{"avg_nll":5.734375,"num_tokens":3.0,"prob":0.22152081799074747,"sum_nll":17.203125},"malignant":{"avg_nll":4.25,"num_tokens":4.0,"prob":0.27141297976627526,"sum_nll":17.0},"other":{"avg_nll":8.1875,"num_tokens":2.0,"prob":0.5070662022429773,"sum_nll":16.375}}'},
 {'uid': 2,
  'response': 'The image shows a large, red and swollen mole on the arm of a person. The mole is covered in small, red and purple veins and appears to be inflamed. There are also several small, red and purple spots on the surface of the mole. The skin around the mole appears to be dry and flaky, with small, white pieces of sk

In [5]:

# --- parse NLL probs and predicted label ---
import json

def parse_nll(response):
    idx = response.rfind('###NLL:')
    if idx == -1:
        return {'prob_malignant': None, 'prob_benign': None, 'prob_other': None, 'pred_label': None}
    try:
        nll = json.loads(response[idx + len('###NLL:'):].strip())
        probs = {lbl: nll[lbl]['prob'] for lbl in ['malignant', 'benign', 'other'] if lbl in nll}
        pred_label = max(probs, key=probs.get) if probs else None
        return {
            'prob_malignant': probs.get('malignant'),
            'prob_benign': probs.get('benign'),
            'prob_other': probs.get('other'),
            'pred_label': pred_label,
        }
    except Exception:
        return {'prob_malignant': None, 'prob_benign': None, 'prob_other': None, 'pred_label': None}


In [6]:

results_df = pd.DataFrame(results).set_index('uid')
nll_df = results_df['response'].apply(parse_nll).apply(pd.Series)
results_df = pd.concat([results_df, nll_df], axis=1)

print(f'\nPredicted label distribution:')
print(results_df['pred_label'].value_counts())
print(f'\nProb summary:')
print(results_df[['prob_malignant', 'prob_benign', 'prob_other']].describe())



Predicted label distribution:
pred_label
malignant    61
other        41
benign        3
Name: count, dtype: int64

Prob summary:
       prob_malignant  prob_benign  prob_other
count      105.000000   105.000000  105.000000
mean         0.485102     0.142884    0.372014
std          0.225606     0.117690    0.252753
min          0.082226     0.005928    0.014005
25%          0.322223     0.064520    0.140323
50%          0.479038     0.115007    0.361810
75%          0.686896     0.202246    0.581482
max          0.919618     0.741734    0.897936


In [7]:
results_df

,response,prob_malignant,prob_benign,prob_other,pred_label
uid,,,,,
1,"This is an image of a person's arm, showing re...",0.271413,0.221521,0.507066,other
2,"The image shows a large, red and swollen mole ...",0.221341,0.128102,0.650557,other
3,This image appears to be a close up view of a ...,0.713243,0.117347,0.169410,malignant
4,This image appears to be a scan of a woman's a...,0.195178,0.134144,0.670678,other
5,This image shows a close up view of a woman's ...,0.290146,0.114514,0.595340,other
...,...,...,...,...,...
101,This is an image of a person's eye with a smal...,0.127944,0.050694,0.821362,other
102,This image shows a close up view of the patien...,0.381345,0.037173,0.581482,other
103,This is an image of a person's arm with a smal...,0.108234,0.040444,0.851323,other


In [ ]:
# # --- save indexed by uid ---
# os.makedirs('./results', exist_ok=True)
# out_path = './results/raw_skingpt4_predictions.csv'

# results_df = pd.DataFrame(results).set_index('uid')
# results_df.to_csv(out_path)
# print(f'Saved {len(results_df)} rows to {out_path}')
# results_df.head()